# A_data: sanity checks for `src/data.py` (Member A)

Run with the **Python (cifar)** kernel, Jupyter started from the repo root.

## Environment check

In [ ]:
import sys, torch, torchvision, numpy, matplotlib, sklearn
print("Python     :", sys.version.split()[0])
print("torch      :", torch.__version__)
print("torchvision:", torchvision.__version__)
print("numpy      :", numpy.__version__)
print("GPU        :", torch.cuda.is_available())

## Cell 1: imports and paths
The notebook sits two folders below the repo root, so `../..` points back to the root.

In [ ]:
import sys
sys.path.append("../..")                # lets Python find the src/ package

from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import torch

from src.data import (get_dataloaders, get_datasets, split_indices,
                      compute_mean_std, CLASSES, CIFAR10_MEAN, CIFAR10_STD)

DATA_DIR = "../../data"                 # one shared data/ folder at the repo root
FIG_DIR = "../../results/ex1/figures"

## Cell 2: download and check the sizes
The first run downloads about 170 MB.

In [ ]:
train_set, val_set, test_set = get_datasets(data_dir=DATA_DIR)
print("train:", len(train_set), " val:", len(val_set), " test:", len(test_set))
assert (len(train_set), len(val_set), len(test_set)) == (45000, 5000, 10000)
# expected: train: 45000  val: 5000  test: 10000

## Cell 3: the split is fixed and has no overlap
Everyone on the team must see the same five validation indices.

In [ ]:
tr_idx, va_idx = split_indices()
tr_idx2, va_idx2 = split_indices()
assert len(set(tr_idx) & set(va_idx)) == 0, "train and val overlap!"
assert (va_idx == va_idx2).all(), "split is not repeatable!"
print("first 5 validation indices:", va_idx[:5])

## Cell 4: class balance in the validation set
Expect roughly 500 per class, within about ±50.

In [ ]:
val_labels = [val_set.dataset.targets[i] for i in val_set.indices]
counts = Counter(val_labels)
for c in range(10):
    print(f"{CLASSES[c]:>10}: {counts[c]}")

## Cell 5: batches have the agreed shape

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(data_dir=DATA_DIR)
images, labels = next(iter(train_loader))
print(images.shape, images.dtype)       # torch.Size([128, 3, 32, 32]) torch.float32
print(labels.shape, labels.dtype)       # torch.Size([128]) torch.int64
print("batches:", len(train_loader), len(val_loader), len(test_loader))   # 352 40 79
print("pixel mean %.3f, std %.3f" % (images.mean(), images.std()))       # close to 0 and 1

## Cell 6: look at the images
Normalised pixels are converted back before display. The figure is saved for the report.

In [ ]:
mean = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
std = torch.tensor(CIFAR10_STD).view(3, 1, 1)

fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for ax, img, label in zip(axes.flat, images[:16], labels[:16]):
    img = (img * std + mean).clamp(0, 1)         # undo Normalize
    ax.imshow(img.permute(1, 2, 0))              # [3,32,32] -> [32,32,3] for matplotlib
    ax.set_title(CLASSES[label.item()], fontsize=8)
    ax.axis("off")
plt.suptitle("CIFAR-10 training samples")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/sample_images.png", dpi=150)
plt.show()

## Cell 7: normalisation values computed from our 45k training images
Should differ from the standard values only in the third decimal place.

In [ ]:
m, s = compute_mean_std(data_dir=DATA_DIR)
print("computed mean:", np.round(m, 4), "  std:", np.round(s, 4))
print("standard mean:", CIFAR10_MEAN, "  std:", CIFAR10_STD)